# 01 · BRONZE — Ingestão IPCA (BCB API)

**Fonte:** Banco Central do Brasil — SGS Série 433 (IPCA mensal)  
**Período:** 01/01/2024 → hoje  
**Destino:** `etl_pos__bronze.ipca` (Delta Lake · overwrite)  
**SPEC:** SPEC_BRONZE.md — schema: data (String), ipca (Double), data_coleta (Timestamp)

> **PRÉ-REQUISITO:** Executar `00.config/config.ipynb` na mesma sessão antes deste notebook.

In [ ]:
# ============================================================
# CELL 1 — Imports e Spark Session
# ============================================================
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import requests
import pandas as pd
from datetime import datetime

spark = SparkSession.builder.getOrCreate()

# Constantes
TARGET_TABLE = "etl_pos__bronze.ipca"
BCB_URL      = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados"
DATA_INICIAL = "01/01/2024"
DATA_FINAL   = datetime.now().strftime("%d/%m/%Y")

print(f"Período: {DATA_INICIAL} → {DATA_FINAL}")
print(f"Destino: {TARGET_TABLE}")

In [ ]:
# ============================================================
# CELL 2 — Extração via BCB API SGS
# ============================================================
params = {
    "formato"     : "json",
    "dataInicial" : DATA_INICIAL,
    "dataFinal"   : DATA_FINAL,
}

print(f"Chamando BCB API...")
response = requests.get(BCB_URL, params=params, timeout=30)
response.raise_for_status()  # Lança exceção se status != 200

raw_data = response.json()
print(f"Registros recebidos: {len(raw_data)}")
print(f"Primeiro registro  : {raw_data[0]}")
print(f"Ultimo registro    : {raw_data[-1]}")

In [ ]:
# ============================================================
# CELL 3 — Conversão pandas → PySpark
# NOTA: pandas usado APENAS como bridge de conversão.
#       Nunca para cálculos (viola disciplina PySpark).
# ============================================================

# Schema explícito — evita inferência incorreta
schema = StructType([
    StructField("data",  StringType(), nullable=False),
    StructField("ipca",  DoubleType(), nullable=True),
])

# Pandas para normalização da string BCB
df_pd = pd.DataFrame(raw_data)
df_pd.columns = ["data", "ipca"]
df_pd["ipca"] = pd.to_numeric(
    df_pd["ipca"].str.replace(",", ".", regex=False),
    errors="coerce"
)

# Criar DataFrame Spark via createDataFrame (padrão CE)
df_spark = spark.createDataFrame(df_pd, schema=schema)

# Adicionar timestamp de ingestão (data_coleta)
df_spark = df_spark.withColumn("data_coleta", F.current_timestamp())

print(f"Schema do DataFrame Spark:")
df_spark.printSchema()
print(f"Total de registros: {df_spark.count()}")

In [ ]:
# ============================================================
# CELL 4 — Preview dos dados
# ============================================================
display(df_spark)

In [ ]:
# ============================================================
# CELL 5 — Gravar na camada Bronze (Delta · overwrite)
# ADR-003: saveAsTable() SEM path no Databricks CE.
#          O CE gerencia o storage internamente.
# ============================================================
(
    df_spark
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Tabela '{TARGET_TABLE}' gravada com sucesso!")

In [ ]:
# ============================================================
# CELL 6 — Validação pós-escrita (critérios TASK-001)
# ============================================================
df_val = spark.table(TARGET_TABLE)
total  = df_val.count()
nulls  = df_val.filter(F.col("ipca").isNull()).count()

print("=" * 50)
print("VALIDAÇÃO — etl_pos__bronze.ipca")
print("=" * 50)
print(f"  Total de registros : {total}")
print(f"  Registros com null : {nulls}")
print(f"  Criterio >= 12 rows: {'OK' if total >= 12 else 'FALHOU'}")
print(f"  Criterio 0 nulls   : {'OK' if nulls == 0 else 'ATENCAO - verificar'}")

display(df_val.orderBy("data"))